In [2]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / 'dataset').exists() and (candidate / 'paths.py').exists():
        candidate_str = str(candidate.resolve())
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)
        break


### Import packages and load data

In [3]:
from pathlib import Path

import pandas as pd
from qdrant_client import QdrantClient
from transformers import AutoModel

from config import (
    QDRANT_ICD_PROCEDURE_COLLECTION_NAME,
    QDRANT_ICD_PROCEDURE_EMBEDDING_MODEL,
    QDRANT_URL,
)
from paths import QDRANT_STORAGE_DIR
from qdrant_collection import Qdrant_Collection, batch_upsert_procedures

from dataset.data import BASE_HOSP

d_icd_procedures_path = "d_icd_procedures.csv"
procedures_path = "procedures_icd.csv"

d_icd_procedures = pd.read_csv(BASE_HOSP.joinpath(d_icd_procedures_path))
procedures = pd.read_csv(BASE_HOSP.joinpath(procedures_path))

### Start the docker client

From `HospitalAgent/`, either start `qdrant/qdrant` via the Docker GUI or run

```
docker run -p 6333:6333 -p 6334:6334 \
  -v "$(pwd)/src/raw/runtime/qdrant/main:/qdrant/storage:z" \
  -e QDRANT__TELEMETRY_DISABLED=true \
  qdrant/qdrant
```
in the terminal (same canonical storage location used by runtime procedure lookup).

For more info, please check out: https://qdrant.tech/documentation/quickstart/.

# Run embedding

- Load Huggingface embedding model
- create feature vectors
- upsert into Qdrant DB

In [ ]:
# skip this cell
# embedding_client = AutoModel.from_pretrained(
#     QDRANT_ICD_PROCEDURE_EMBEDDING_MODEL, trust_remote_code=True
# )

# qdrant_client = QdrantClient(url=QDRANT_URL)
# collection = Qdrant_Collection(
#     qdrant_client,
#     embedding_client,
#     QDRANT_ICD_PROCEDURE_COLLECTION_NAME,
#     QDRANT_ICD_PROCEDURE_EMBEDDING_MODEL,
# )

# batch_upsert_procedures(
#     collection, d_icd_procedures.to_dict(orient="records"), max_batch_size=200
# )

# Run embedding against the running Qdrant server

Connects via `QDRANT_URL` (the Docker server started by `05_start_qdrant.sh`) instead of qdrant-client's local
embedded file mode. That server already has `../raw/runtime/qdrant/main` mounted as its storage, so writing to
the same path from a second, separate embedded client caused a conflict: the collection got created (schema
only) but the points never durably landed — `points_count` stayed 0 on the server even though this notebook's
own in-process client could see its own writes. Going through the server directly avoids the two-writer
conflict.

In [4]:
embedding_client = AutoModel.from_pretrained(
    QDRANT_ICD_PROCEDURE_EMBEDDING_MODEL, trust_remote_code=True
)

qdrant_client = QdrantClient(url=QDRANT_URL)
collection = Qdrant_Collection(
    qdrant_client,
    embedding_client,
    QDRANT_ICD_PROCEDURE_COLLECTION_NAME,
    QDRANT_ICD_PROCEDURE_EMBEDDING_MODEL,
)

batch_upsert_procedures(
    collection, d_icd_procedures.to_dict(orient="records"), max_batch_size=200
)

flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn i

In [5]:
collection.search("appendectomy", query_filter=None, top_k=10)

QueryResponse(points=[ScoredPoint(id='2bfc35aa-e5bf-4c0c-a3cd-2c0fab62911b', version=379, score=0.9536556, payload={'icd_code': '4709', 'icd_version': 9, 'long_title': 'Other appendectomy'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='e3f1f664-d881-4c03-97a7-de62503b5d2f', version=812, score=0.9536556, payload={'icd_code': '4709', 'icd_version': 9, 'long_title': 'Other appendectomy'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='e77be921-7c5b-40c8-a528-35bb6dddf09e', version=379, score=0.9083955, payload={'icd_code': '4791', 'icd_version': 9, 'long_title': 'Appendicostomy'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='b9436acb-dc6e-4e2a-abfd-6da9137f217b', version=812, score=0.9083955, payload={'icd_code': '4791', 'icd_version': 9, 'long_title': 'Appendicostomy'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='151284ab-893a-4033-b668-afd7cad3caa6', version=812, score=0.8922094, payload={'icd_code': '4701', 'icd

In [6]:
results = collection.search("appendectomy", query_filter=None, top_k=10)
points = results.points if hasattr(results, "points") else results[0]

for idx, point in enumerate(points, start=1):
    icd_code = point.payload.get("icd_code")
    title = point.payload.get("long_title")
    score = point.score
    
    print(f"{idx:2d}. [{icd_code}] {title} (Score: {score:.4f})")

 1. [4709] Other appendectomy (Score: 0.9537)
 2. [4709] Other appendectomy (Score: 0.9537)
 3. [4791] Appendicostomy (Score: 0.9084)
 4. [4791] Appendicostomy (Score: 0.9084)
 5. [4701] Laparoscopic appendectomy (Score: 0.8922)
 6. [4701] Laparoscopic appendectomy (Score: 0.8922)
 7. [4719] Other incidental appendectomy (Score: 0.8806)
 8. [4719] Other incidental appendectomy (Score: 0.8806)
 9. [0DTJ0ZZ] Resection of Appendix, Open Approach (Score: 0.8486)
10. [0DTJ0ZZ] Resection of Appendix, Open Approach (Score: 0.8486)
